# Supervised Learning. Decision Trees and ensembles

Метрика: Gini = 2 * ROC AUC - 1

In [66]:
import numpy as np
import pandas as pd
import category_encoders as ce
from sklearn.metrics import roc_auc_score
from sklearn.tree import DecisionTreeClassifier
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier


def gini(y_true, y_score):
    return 2 * roc_auc_score(y_true, y_score) - 1

## 1. Данные и разбиение train / valid / test

В `data/test.csv` из соревнования нет целевой переменной `IsBadBuy`, поэтому все три выборки строятся из `data/train.csv` по полю `PurchDate`: первые 33% — train, средние 33% — valid, последние 33% — test. Граница проходит по дате, поэтому выполняется train.PurchDate < valid.PurchDate < test.PurchDate.

In [67]:
df = pd.read_csv('data/train.csv')
df['PurchDate'] = pd.to_datetime(df['PurchDate'], format='%m/%d/%Y')
df = df.sort_values('PurchDate').reset_index(drop=True)

n = len(df)
d1 = df['PurchDate'].iloc[n // 3]
d2 = df['PurchDate'].iloc[2 * n // 3]

train = df[df['PurchDate'] < d1]
valid = df[(df['PurchDate'] >= d1) & (df['PurchDate'] < d2)]
test = df[df['PurchDate'] >= d2]

print(len(train), len(valid), len(test))
print(train['PurchDate'].max() < valid['PurchDate'].min(), valid['PurchDate'].max() < test['PurchDate'].min())

24232 24414 24337
True True


Категориальные признаки кодируются `CountEncoder` (обучается только на train). Он корректно обрабатывает новые категории в valid/test, которых не было в train (им ставится 0). Пропуски в числовых признаках заполняются медианами train.

In [68]:
target = 'IsBadBuy'
drop_cols = ['RefId', 'IsBadBuy', 'PurchDate']
cat_cols = ['Auction', 'Make', 'Model', 'Trim', 'SubModel', 'Color', 'Transmission',
            'WheelType', 'Nationality', 'Size', 'TopThreeAmericanName',
            'PRIMEUNIT', 'AUCGUART', 'VNST']
num_cols = [c for c in df.columns if c not in drop_cols + cat_cols]


def split_xy(d):
    X = d[cat_cols + num_cols].copy()
    X[cat_cols] = X[cat_cols].fillna('NA').astype(str)
    return X, d[target].values


X_train_raw, y_train = split_xy(train)
X_valid_raw, y_valid = split_xy(valid)
X_test_raw, y_test = split_xy(test)

medians = X_train_raw[num_cols].median()
for X in (X_train_raw, X_valid_raw, X_test_raw):
    X[num_cols] = X[num_cols].fillna(medians)

encoder = ce.CountEncoder(cols=cat_cols)
encoder.fit(X_train_raw)
Xtr = encoder.transform(X_train_raw).fillna(0).values.astype(float)
Xva = encoder.transform(X_valid_raw).fillna(0).values.astype(float)
Xte = encoder.transform(X_test_raw).fillna(0).values.astype(float)
Xtr.shape, Xva.shape, Xte.shape

((24232, 31), (24414, 31), (24337, 31))

## 2. Decision Tree Classifier / Regressor

`Node` хранит данные узла, считает Gini (классификация) и стандартное отклонение (регрессия), имеет ссылки на левого и правого потомка.

`find_best_split` — перебор всех порогов по признакам (через сортировку и кумулятивные суммы). `find_random_split` — один случайный порог на признак (Extra Randomized Tree).

In [69]:
class Node:
    def __init__(self, X, y, max_depth=0):
        self.X = X
        self.y = y
        self.max_depth = max_depth
        self.value = y.mean()
        self.feature = None
        self.threshold = None
        self.left = None
        self.right = None

    def gini(self):
        _, counts = np.unique(self.y, return_counts=True)
        p = counts / counts.sum()
        return 1 - np.sum(p ** 2)

    def std(self):
        return self.y.std()

    def is_leaf(self):
        return self.left is None


def split_cost(s_l, q_l, n_l, s, q, n, criterion):
    # s, q - сумма y и y^2 в узле; s_l, q_l, n_l - то же для левой части
    s_r, q_r, n_r = s - s_l, q - q_l, n - n_l
    if criterion == 'gini':
        p_l, p_r = s_l / n_l, s_r / n_r
        return n_l * 2 * p_l * (1 - p_l) + n_r * 2 * p_r * (1 - p_r)
    # 'std': взвешенная дисперсия (std^2) потомков = MSE
    var_l = np.maximum(q_l / n_l - (s_l / n_l) ** 2, 0)
    var_r = np.maximum(q_r / n_r - (s_r / n_r) ** 2, 0)
    return n_l * var_l + n_r * var_r


def find_best_split(X, y, criterion, feature_ids, min_samples_leaf):
    n, s, q = len(y), y.sum(), (y ** 2).sum()
    best = (None, None, np.inf)
    for f in feature_ids:
        order = np.argsort(X[:, f], kind='mergesort')
        x, ys = X[order, f], y[order]
        s_l = np.cumsum(ys)[:-1]
        q_l = np.cumsum(ys ** 2)[:-1]
        n_l = np.arange(1, n)
        ok = (x[:-1] != x[1:]) & (n_l >= min_samples_leaf) & (n - n_l >= min_samples_leaf)
        if not ok.any():
            continue
        cost = split_cost(s_l[ok], q_l[ok], n_l[ok], s, q, n, criterion)
        i = np.argmin(cost)
        if cost[i] < best[2]:
            idx = np.where(ok)[0][i]
            best = (f, (x[idx] + x[idx + 1]) / 2, cost[i])
    return best


def find_random_split(X, y, criterion, feature_ids, min_samples_leaf, rng):
    n, s, q = len(y), y.sum(), (y ** 2).sum()
    best = (None, None, np.inf)
    for f in feature_ids:
        lo, hi = X[:, f].min(), X[:, f].max()
        if lo == hi:
            continue
        t = rng.uniform(lo, hi)
        mask = X[:, f] <= t
        n_l = mask.sum()
        if n_l < min_samples_leaf or n - n_l < min_samples_leaf:
            continue
        cost = split_cost(y[mask].sum(), (y[mask] ** 2).sum(), n_l, s, q, n, criterion)
        if cost < best[2]:
            best = (f, t, cost)
    return best

In [70]:
class BaseDecisionTree:
    criterion = None

    def __init__(self, max_depth=5, min_samples_leaf=1, max_features=None,
                 splitter='best', random_state=None):
        self.max_depth = max_depth
        self.min_samples_leaf = min_samples_leaf
        self.max_features = max_features
        self.splitter = splitter
        self.random_state = random_state

    def _n_features(self, n):
        if self.max_features is None:
            return n
        if self.max_features == 'sqrt':
            return max(1, int(np.sqrt(n)))
        if isinstance(self.max_features, float):
            return max(1, int(self.max_features * n))
        return min(n, self.max_features)

    def fit(self, X, y):
        X, y = np.asarray(X, float), np.asarray(y, float)
        self.rng = np.random.default_rng(self.random_state)
        self.root = self._build(Node(X, y, 0))
        return self

    def _build(self, node):
        impurity = node.gini() if self.criterion == 'gini' else node.std()
        if (node.max_depth >= self.max_depth or impurity == 0
                or len(node.y) < 2 * self.min_samples_leaf):
            node.X = node.y = None
            return node

        n_feat = node.X.shape[1]
        feats = self.rng.choice(n_feat, self._n_features(n_feat), replace=False)
        if self.splitter == 'best':
            f, t, _ = find_best_split(node.X, node.y, self.criterion, feats, self.min_samples_leaf)
        else:
            f, t, _ = find_random_split(node.X, node.y, self.criterion, feats,
                                        self.min_samples_leaf, self.rng)
        if f is None:
            node.X = node.y = None
            return node

        mask = node.X[:, f] <= t
        node.feature, node.threshold = f, t
        node.left = self._build(Node(node.X[mask], node.y[mask], node.max_depth + 1))
        node.right = self._build(Node(node.X[~mask], node.y[~mask], node.max_depth + 1))
        node.X = node.y = None
        return node

    def _predict_value(self, X):
        X = np.asarray(X, float)
        out = np.empty(len(X))
        self._fill(self.root, X, np.arange(len(X)), out)
        return out

    def _fill(self, node, X, idx, out):
        if node.is_leaf():
            out[idx] = node.value
            return
        m = X[idx, node.feature] <= node.threshold
        self._fill(node.left, X, idx[m], out)
        self._fill(node.right, X, idx[~m], out)


class DecisionTreeClassifier(BaseDecisionTree):
    criterion = 'gini'

    def predict_proba(self, X):
        p = self._predict_value(X)
        return np.column_stack([1 - p, p])

    def predict(self, X):
        return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)


class DecisionTreeRegressor(BaseDecisionTree):
    criterion = 'std'

    def predict(self, X):
        return self._predict_value(X)

## 3. Собственное дерево: Gini на валидации (нужно >= 0.1)

In [71]:
for d in [3, 5, 7, 9]:
    m = DecisionTreeClassifier(max_depth=d, min_samples_leaf=50).fit(Xtr, y_train)
    print(d, round(gini(y_valid, m.predict_proba(Xva)[:, 1]), 4))

3 0.4331
5 0.4445
7 0.4429
9 0.4239


In [72]:
model = DecisionTreeClassifier(max_depth=7, min_samples_leaf=50)
model.fit(Xtr, y_train)
my_tree_gini = gini(y_valid, model.predict_proba(Xva)[:, 1])
print('My DecisionTree valid Gini:', round(my_tree_gini, 4))

extra = DecisionTreeClassifier(max_depth=7, min_samples_leaf=50, splitter='random', random_state=42)
extra.fit(Xtr, y_train)
print('My Extra Randomized Tree valid Gini:', round(gini(y_valid, extra.predict_proba(Xva)[:, 1]), 4))

My DecisionTree valid Gini: 0.4443
My Extra Randomized Tree valid Gini: 0.4334


## 4. sklearn DecisionTreeClassifier

In [73]:
sk_tree = DecisionTreeClassifier(max_depth=7, min_samples_leaf=50, random_state=42).fit(Xtr, y_train)
print('sklearn valid Gini:', round(gini(y_valid, sk_tree.predict_proba(Xva)[:, 1]), 4))
print('my tree valid Gini:', round(my_tree_gini, 4))

sklearn valid Gini: 0.4494
my tree valid Gini: 0.4443


**Вывод.** sklearn реализует тот же алгоритм CART с критерием Gini, поэтому при одинаковых `max_depth` и `min_samples_leaf` качество практически совпадает. Небольшие различия возникают из-за выбора между равными по качеству разбиениями (sklearn перемешивает признаки случайно) и численных деталей. Главное преимущество sklearn — скорость (реализация на Cython), а не качество.

## 5. RandomForestClassifier (нужно >= 0.15)

In [74]:
class RandomForestClassifier:
    def __init__(self, n_estimators=50, max_depth=10, max_features='sqrt', min_samples_leaf=20,
                 bootstrap=True, splitter='best', random_state=None):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.max_features = max_features
        self.min_samples_leaf = min_samples_leaf
        self.bootstrap = bootstrap
        self.splitter = splitter
        self.random_state = random_state

    def fit(self, X, y):
        X, y = np.asarray(X, float), np.asarray(y, float)
        rng = np.random.default_rng(self.random_state)
        n = len(y)
        self.trees = []
        for _ in range(self.n_estimators):
            idx = rng.integers(0, n, n) if self.bootstrap else np.arange(n)
            tree = DecisionTreeClassifier(max_depth=self.max_depth,
                                          min_samples_leaf=self.min_samples_leaf,
                                          max_features=self.max_features,
                                          splitter=self.splitter,
                                          random_state=int(rng.integers(1e9)))
            self.trees.append(tree.fit(X[idx], y[idx]))
        return self

    def predict_proba(self, X):
        p = np.mean([t.predict_proba(X)[:, 1] for t in self.trees], axis=0)
        return np.column_stack([1 - p, p])

    def predict(self, X):
        return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)


rf = RandomForestClassifier(n_estimators=50, max_depth=10, min_samples_leaf=20, random_state=42)
rf.fit(Xtr, y_train)
print('RandomForest valid Gini:', round(gini(y_valid, rf.predict_proba(Xva)[:, 1]), 4))

RandomForest valid Gini: 0.4717


## 6. GBDT на основе своего DecisionTreeRegressor

BCE: L = -[y log p + (1-y) log(1-p)], p = sigmoid(F). Градиент по F: dL/dF = p - y. Каждое следующее дерево обучается на антиградиенте y - p, вычисленном по предсказаниям уже построенных деревьев.

In [75]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))


class GBDTClassifier:
    def __init__(self, number_of_trees=100, max_depth=3, max_features=None,
                 learning_rate=0.1, min_samples_leaf=20, random_state=None):
        self.number_of_trees = number_of_trees
        self.max_depth = max_depth
        self.max_features = max_features
        self.learning_rate = learning_rate
        self.min_samples_leaf = min_samples_leaf
        self.random_state = random_state

    def fit(self, X, y):
        X, y = np.asarray(X, float), np.asarray(y, float)
        rng = np.random.default_rng(self.random_state)
        p0 = np.clip(y.mean(), 1e-6, 1 - 1e-6)
        self.init = np.log(p0 / (1 - p0))
        F = np.full(len(y), self.init)
        self.trees = []
        for _ in range(self.number_of_trees):
            grad = sigmoid(F) - y  # градиент BCE по F
            tree = DecisionTreeRegressor(max_depth=self.max_depth,
                                         min_samples_leaf=self.min_samples_leaf,
                                         max_features=self.max_features,
                                         random_state=int(rng.integers(1e9)))
            tree.fit(X, -grad)
            F += self.learning_rate * tree.predict(X)
            self.trees.append(tree)
        return self

    def predict_proba(self, X):
        F = self.init + self.learning_rate * np.sum([t.predict(X) for t in self.trees], axis=0)
        p = sigmoid(F)
        return np.column_stack([1 - p, p])

    def predict(self, X):
        return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)


gbdt = GBDTClassifier(number_of_trees=100, max_depth=3, max_features=0.8, learning_rate=0.1, random_state=42)
gbdt.fit(Xtr, y_train)
print('My GBDT valid Gini:', round(gini(y_valid, gbdt.predict_proba(Xva)[:, 1]), 4))

My GBDT valid Gini: 0.4704


## 7. LightGBM, CatBoost, XGBoost

Подбор гиперпараметров — по валидации (с early stopping на valid). Тестовая выборка не используется.

In [76]:
best_lgb, best_lgb_gini = None, -1
for num_leaves in [15, 31, 63]:
    m = lgb.LGBMClassifier(n_estimators=2000, learning_rate=0.03, num_leaves=num_leaves,
                           min_child_samples=50, subsample=0.8, subsample_freq=1,
                           colsample_bytree=0.8, random_state=42, verbose=-1)
    m.fit(Xtr, y_train, eval_X=Xva, eval_y=y_valid, eval_metric='auc',
          callbacks=[lgb.early_stopping(100, verbose=False)])
    g = gini(y_valid, m.predict_proba(Xva)[:, 1])
    print('num_leaves', num_leaves, round(g, 4))
    if g > best_lgb_gini:
        best_lgb, best_lgb_gini = m, g

num_leaves 15 0.4923
num_leaves 31 0.495
num_leaves 63 0.4892


In [77]:
best_xgb, best_xgb_gini = None, -1
for max_depth in [3, 5, 7]:
    m = xgb.XGBClassifier(n_estimators=2000, learning_rate=0.03, max_depth=max_depth,
                          subsample=0.8, colsample_bytree=0.8, min_child_weight=5,
                          eval_metric='auc', early_stopping_rounds=100, random_state=42)
    m.fit(Xtr, y_train, eval_set=[(Xva, y_valid)], verbose=False)
    g = gini(y_valid, m.predict_proba(Xva)[:, 1])
    print('max_depth', max_depth, round(g, 4))
    if g > best_xgb_gini:
        best_xgb, best_xgb_gini = m, g

# DART mode
xgb_dart = xgb.XGBClassifier(booster='dart', rate_drop=0.1, skip_drop=0.5, n_estimators=300,
                             learning_rate=0.05, max_depth=5, subsample=0.8,
                             colsample_bytree=0.8, eval_metric='auc', random_state=42)
xgb_dart.fit(Xtr, y_train)
xgb_dart_gini = gini(y_valid, xgb_dart.predict_proba(Xva)[:, 1])
print('XGBoost DART', round(xgb_dart_gini, 4))

max_depth 3 0.4866
max_depth 5 0.4958
max_depth 7 0.4941
XGBoost DART 0.4951


In [78]:
best_cb, best_cb_gini = None, -1
for depth in [4, 6, 8]:
    m = CatBoostClassifier(iterations=2000, learning_rate=0.03, depth=depth,
                           eval_metric='AUC', od_type='Iter', od_wait=200,
                           random_seed=42, verbose=0)
    m.fit(X_train_raw, y_train, cat_features=cat_cols, eval_set=(X_valid_raw, y_valid))
    g = gini(y_valid, m.predict_proba(X_valid_raw)[:, 1])
    print('depth', depth, round(g, 4))
    if g > best_cb_gini:
        best_cb, best_cb_gini = m, g

depth 4 0.4905
depth 6 0.4926
depth 8 0.4915


In [79]:
enc_data = (Xtr, Xva, Xte)
raw_data = (X_train_raw, X_valid_raw, X_test_raw)
models = {
    'LightGBM': (best_lgb, enc_data),
    'XGBoost': (best_xgb, enc_data),
    'XGBoost DART': (xgb_dart, enc_data),
    'CatBoost': (best_cb, raw_data),
}
valid_scores = {name: gini(y_valid, m.predict_proba(d[1])[:, 1]) for name, (m, d) in models.items()}
pd.Series(valid_scores).sort_values(ascending=False)

XGBoost         0.495765
XGBoost DART    0.495065
LightGBM        0.494994
CatBoost        0.492620
dtype: float64

**Ключевые различия.**

- **XGBoost** — деревья растут по уровням (level-wise), регуляризация L1/L2 весов листьев, используется вторая производная (Newton step). Категориальные признаки нужно кодировать заранее. **DART** (Dropouts meet Multiple Additive Regression Trees) — на каждой итерации случайно «выключается» часть уже построенных деревьев (`rate_drop`), новое дерево обучается на остатках без них, после чего веса нормализуются. Это борется с ситуацией, когда первые деревья вносят основной вклад, а последующие лишь слабо корректируют, и уменьшает переобучение; минус — медленнее и нельзя нормально использовать early stopping.
- **LightGBM** — гистограммный поиск разбиений, рост дерева по листьям (leaf-wise, `num_leaves`), GOSS и EFB для ускорения. Очень быстрый, но на небольших данных leaf-wise рост легче переобучается.
- **CatBoost** — симметричные (oblivious) деревья, ordered boosting против сдвига предсказаний. **Категориальные признаки** подаются как есть: они кодируются ordered target statistics — для каждого объекта среднее таргета по категории считается только по объектам, стоящим раньше него в случайной перестановке (со сглаживанием prior), что убирает утечку таргета. Также автоматически строятся комбинации категориальных признаков.

**Какая модель лучше.** Лучшая по таблице выше. Обычно на этих данных выигрывает CatBoost: здесь много категориальных признаков с высокой кардинальностью (Model, Trim, SubModel, VNST), и target statistics с комбинациями признаков несут больше информации, чем count encoding, а симметричные деревья и ordered boosting дают сильную регуляризацию на шумной задаче.

## 8. Лучшая модель на test

In [80]:
best_name = max(valid_scores, key=valid_scores.get)
best_model, (d_tr, d_va, d_te) = models[best_name]
print('Best model:', best_name)
print('Train Gini:', round(gini(y_train, best_model.predict_proba(d_tr)[:, 1]), 4))
print('Valid Gini:', round(gini(y_valid, best_model.predict_proba(d_va)[:, 1]), 4))
print('Test Gini: ', round(gini(y_test, best_model.predict_proba(d_te)[:, 1]), 4))

Best model: XGBoost
Train Gini: 0.6094
Valid Gini: 0.4958
Test Gini:  0.5064


**Вывод.** Train Gini заметно выше valid и test — это нормально для бустинга (модель частично запоминает обучающую выборку). Небольшое падение valid → test ожидаемо: гиперпараметры и число деревьев (early stopping) подбирались на valid, поэтому оценка на valid слегка оптимистична; кроме того, test дальше по времени от train, и распределение данных сдвигается. Если падение valid → test небольшое, модель обобщается и сильного переобучения нет; разрыв train/valid говорит о лёгком переобучении, которое контролируется early stopping и регуляризацией.

## 9*. ExtraTreesClassifier (нужно >= 0.12)

In [81]:
class ExtraTreesClassifier(RandomForestClassifier):
    def __init__(self, n_estimators=50, max_depth=10, max_features='sqrt',
                 min_samples_leaf=20, random_state=None):
        super().__init__(n_estimators=n_estimators, max_depth=max_depth,
                         max_features=max_features, min_samples_leaf=min_samples_leaf,
                         bootstrap=False, splitter='random', random_state=random_state)


et = ExtraTreesClassifier(n_estimators=100, max_depth=12, max_features=0.5, min_samples_leaf=20, random_state=42)
et.fit(Xtr, y_train)
print('ExtraTrees valid Gini:', round(gini(y_valid, et.predict_proba(Xva)[:, 1]), 4))

ExtraTrees valid Gini: 0.4854
